## 1. Download Dataset from Google Drive
Uses the official Google Drive API to bypass folder size limits and download all files reliably.

In [1]:
from google.colab import auth
from googleapiclient.discovery import build
import io
import os
from googleapiclient.http import MediaIoBaseDownload

# 1. Authenticate user
auth.authenticate_user()

# 2. Build Drive API Service
drive_service = build('drive', 'v3')

folder_id = '1Cl2m8yBkS7NF2l2_6V-rW8RRv0EfBOXD'
output_dir = 'clean_reviews.parquet'

os.makedirs(output_dir, exist_ok=True)

# 3. List and Download Files
query = f"'{folder_id}' in parents and trashed=false"
results = drive_service.files().list(q=query, fields="nextPageToken, files(id, name)").execute()
items = results.get('files', [])

if not items:
    print('No files found in the folder.')
else:
    print(f'Found {len(items)} files. Starting download...')
    for item in items:
        file_id = item['id']
        file_name = item['name']
        file_path = os.path.join(output_dir, file_name)
        
        print(f"Downloading {file_name}...")
        request = drive_service.files().get_media(fileId=file_id)
        fh = io.FileIO(file_path, 'wb')
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while done is False:
            status, done = downloader.next_chunk()
    print("\nAll downloads complete!")

Found 100 files. Starting download...

All downloads complete!


## 2. Check Folder Size
Calculates the total size of the `clean_reviews.parquet` folder in human-readable format (MB/GB).

In [2]:
import os

def get_dir_size(path='.'):
    total_size = 0
    if os.path.exists(path):
        for dirpath, dirnames, filenames in os.walk(path):
            for f in filenames:
                fp = os.path.join(dirpath, f)
                if not os.path.islink(fp):
                    total_size += os.path.getsize(fp)
    return total_size

folder_size_bytes = get_dir_size('clean_reviews.parquet')

print(f"Dataset Folder: clean_reviews.parquet")
print(f"Total Size (Bytes): {folder_size_bytes:,}")
print(f"Total Size (MB): {folder_size_bytes / (1024 * 1024):.2f} MB")
print(f"Total Size (GB): {folder_size_bytes / (1024 * 1024 * 1024):.4f} GB")

Dataset Folder: clean_reviews.parquet
Total Size (Bytes): 6,490,683,036
Total Size (MB): 6190.00 MB
Total Size (GB): 6.0449 GB


## 3. Verify CRC Checksums
Hadoop/Spark generated `.crc` files are binary files containing chunked CRC32 checksums. This block reads the binary header, iterates chunk-by-chunk through the `.parquet` file, computes the CRC32 hash, and verifies it against the `.crc` file.

In [3]:
import os
import glob
import struct
import zlib

dataset_dir = 'clean_reviews.parquet'
parquet_files = glob.glob(f'{dataset_dir}/**/*.parquet', recursive=True)

success_count = 0
total_count = len(parquet_files)

for data_path in parquet_files:
    dir_name = os.path.dirname(data_path)
    base_name = os.path.basename(data_path)
    
    # Hadoop CRC files follow the pattern ".<filename>.crc"
    crc_path = os.path.join(dir_name, f'.{base_name}.crc')
    
    if not os.path.exists(crc_path):
        print(f"[{base_name}] Missing corresponding .crc file.")
        continue
        
    try:
        with open(crc_path, 'rb') as f_crc:
            # Spark/HDFS ChecksumFileSystem uses "crc\x00" as the magic 4-bytes
            magic = f_crc.read(4)
            if magic != b'crc\x00':
                print(f"[{base_name}] Unknown CRC format magic. Cannot parse.")
                continue
            
            # Unpack the 4-byte chunk size (usually 512 bytes)
            bytes_per_chunk_data = f_crc.read(4)
            bytes_per_chunk = struct.unpack('>I', bytes_per_chunk_data)[0]
            
            # Read all expected 4-byte CRC32 integers for each chunk
            crc_values = []
            while True:
                chunk_crc_data = f_crc.read(4)
                if not chunk_crc_data:
                    break
                if len(chunk_crc_data) == 4:
                    crc_values.append(struct.unpack('>I', chunk_crc_data)[0])
                    
        # Check chunks against CRC values
        match = True
        with open(data_path, 'rb') as f_data:
            for expected_crc in crc_values:
                data_chunk = f_data.read(bytes_per_chunk)
                if not data_chunk:
                    break
                
                # Calculate CRC32 of chunk & mask with 0xFFFFFFFF to handle signed integers across platforms
                actual_crc = zlib.crc32(data_chunk) & 0xFFFFFFFF
                if actual_crc != expected_crc:
                    match = False
                    break
                    
        if match:
            success_count += 1
            print(f"[{base_name}] CRC PASS")
        else:
            print(f"[{base_name}] CRC FAIL (Checksum mismatch)")
            
    except Exception as e:
        print(f"[{base_name}] Error during verification: {e}")

# Summary Report
if total_count > 0:
    match_rate = (success_count / total_count) * 100
    print("\n" + "="*50)
    print(f"{'-'*15} CRC VERIFICATION {'-'*15}")
    print(f"Parquet Files Found      : {total_count}")
    print(f"Successfully Matched CRC : {success_count}")
    print(f"Overall Match Rate       : {match_rate:.2f}%")
    print("="*50 + "\n")
else:
    print("No parquet files were found in the dataset directory.")

[part-00037-5447ecee-fb4f-4738-bf0c-817cf630f694-c000.snappy.parquet] CRC PASS
[part-00040-5447ecee-fb4f-4738-bf0c-817cf630f694-c000.snappy.parquet] CRC PASS
[part-00053-5447ecee-fb4f-4738-bf0c-817cf630f694-c000.snappy.parquet] CRC PASS
[part-00049-5447ecee-fb4f-4738-bf0c-817cf630f694-c000.snappy.parquet] CRC PASS
[part-00012-5447ecee-fb4f-4738-bf0c-817cf630f694-c000.snappy.parquet] CRC PASS
[part-00033-5447ecee-fb4f-4738-bf0c-817cf630f694-c000.snappy.parquet] CRC PASS
[part-00036-5447ecee-fb4f-4738-bf0c-817cf630f694-c000.snappy.parquet] CRC PASS
[part-00055-5447ecee-fb4f-4738-bf0c-817cf630f694-c000.snappy.parquet] CRC PASS
[part-00025-5447ecee-fb4f-4738-bf0c-817cf630f694-c000.snappy.parquet] CRC PASS
[part-00045-5447ecee-fb4f-4738-bf0c-817cf630f694-c000.snappy.parquet] CRC PASS
[part-00011-5447ecee-fb4f-4738-bf0c-817cf630f694-c000.snappy.parquet] CRC PASS
[part-00058-5447ecee-fb4f-4738-bf0c-817cf630f694-c000.snappy.parquet] CRC PASS
[part-00038-5447ecee-fb4f-4738-bf0c-817cf630f694-c00

## 4. Setup PySpark for Scalable EDA
Installs PySpark and initializes a SparkSession. PySpark uses lazy evaluation and processed data in a distributed fashion, preventing "Out of Memory" errors on large datasets.

In [4]:
!pip install -q pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialize Spark Session (Memory-efficient)
spark = SparkSession.builder \
    .appName("LargeScaleEDA") \
    .master("local[*]") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("Spark Session Started!")

Spark Session Started!


## 5. Memory-Efficient EDA with PySpark
Loads the dataset using Spark's native Parquet reader and performs exploratory analysis without loading all records into RAM at once.

In [7]:
dataset_path = 'clean_reviews.parquet'

# Load data (Lazy loading)
spark_df = spark.read.parquet(dataset_path)

print("--- Dataset Schema ---")
spark_df.printSchema()

rowCount = spark_df.count()
print(f"\nTotal Rows: {rowCount:,}")
print(f"Total Columns: {len(spark_df.columns)}")

print("\n--- First 5 Rows ---")
spark_df.show(5)



--- Dataset Schema ---
root
 |-- asin: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- title: string (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- verified_purchase: boolean (nullable = true)


Total Rows: 35,388,706
Total Columns: 8

--- First 5 Rows ---
+----------+-----------+------+--------------------+--------------------+-------------------+------------+-----------------+
|      asin|parent_asin|rating|               title|                text|          timestamp|helpful_vote|verified_purchase|
+----------+-----------+------+--------------------+--------------------+-------------------+------------+-----------------+
|B07WT32GNK| B07WT32GNK|   1.0|did not work out ...|   title says it all|2022-10-05 16:36:45|           0|             true|
|B00CYX54C0| B00CYX54C0|   4.0|      such potential|i really like the...|2014-04-0

In [8]:
print("\n--- Missing Values Count ---")
missing_counts = spark_df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in spark_df.columns])
missing_counts.show()
 



--- Missing Values Count ---
+----+-----------+------+-----+----+---------+------------+-----------------+
|asin|parent_asin|rating|title|text|timestamp|helpful_vote|verified_purchase|
+----+-----------+------+-----+----+---------+------------+-----------------+
|   0|          0|     0|    0|   0|        0|           0|                0|
+----+-----------+------+-----+----+---------+------------+-----------------+



In [9]:
print("\n--- Column Statistics (Numerical) ---")
# Get the list of column names and their data types
numeric_cols = [col for col, dtype in spark_df.dtypes if dtype in ['int', 'double', 'float', 'bigint', 'decimal']]
spark_df[numeric_cols].describe().show()




--- Column Statistics (Numerical) ---
+-------+------------------+-----------------+
|summary|            rating|     helpful_vote|
+-------+------------------+-----------------+
|  count|          35388706|         35388706|
|   mean| 4.099750383639345|1.036521143214448|
| stddev|1.4119360406421981| 18.3864428200182|
|    min|               1.0|               -4|
|    max|               5.0|            32948|
+-------+------------------+-----------------+



In [10]:
print("\n--- Data Types Check ---")
for col, dtype in spark_df.dtypes:
    print(f"{col}: {dtype}")


--- Data Types Check ---
asin: string
parent_asin: string
rating: double
title: string
text: string
timestamp: timestamp
helpful_vote: bigint
verified_purchase: boolean


In [11]:
numeric_cols

['rating', 'helpful_vote']

## 6. Stratified Sampling (5,000 Rows)
Samples exactly 5,000 random rows without loading all data into memory, maintaining the target distribution:
- 50% for 1 and 2 star ratings (2,500 rows)
- 20% for 3 star ratings (1,000 rows)
- 30% for 4 and 5 star ratings (1,500 rows)

In [12]:
from pyspark.sql.functions import col, rand

# IMPORTANT: Verify the actual name of your rating column from the schema above 
rating_col = 'rating' 

# Subsets based on strata
df_1_2 = spark_df.filter(col(rating_col).isin([1, 2]))
df_3 = spark_df.filter(col(rating_col) == 3)
df_4_5 = spark_df.filter(col(rating_col).isin([4, 5]))

# Target sample sizing per strata
n_1_2 = 2500 # 50% of 5000
n_3 = 1000   # 20% of 5000
n_4_5 = 1500 # 30% of 5000

def get_exact_sample(df, target_n):
    total = df.count()
    if total == 0: return df
    if total <= target_n: return df
    fraction = min(1.0, (target_n * 1.2) / total)
    return df.sample(withReplacement=False, fraction=fraction, seed=42).limit(target_n)

sample_1_2 = get_exact_sample(df_1_2, n_1_2)
sample_3 = get_exact_sample(df_3, n_3)
sample_4_5 = get_exact_sample(df_4_5, n_4_5)

stratified_sample_df = sample_1_2.unionAll(sample_3).unionAll(sample_4_5).orderBy(rand(seed=42))
stratified_sample_df.cache()
print("Stratified sample successfully cached to memory.")
stratified_sample_df.groupBy(rating_col).count().orderBy(rating_col).show()

Stratified sample successfully cached to memory.
+------+-----+
|rating|count|
+------+-----+
|   1.0| 1755|
|   2.0|  745|
|   3.0| 1000|
|   4.0|  256|
|   5.0| 1244|
+------+-----+



## 7. Export Stratified Sample to Google Drive
Mounts Google Drive, ensures the target folder exists, and saves the 5,000-row sample to `sampled_dataset.csv` in `My Drive > Amazon_2023_EDA`.

In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define target directory and file name
drive_folder = '/content/drive/My Drive/Amazon_2023_EDA'
file_name = 'sampled_dataset.csv'
full_path = os.path.join(drive_folder, file_name)

# 3. Create folder if it doesn't exist
if not os.path.exists(drive_folder):
    os.makedirs(drive_folder)
    print(f"Folder created: {drive_folder}")

# 4. Save PySpark DataFrame to CSV
# (For 5,000 rows, toPandas() is safe and provides a single, clean CSV file)
print(f"Saving to {full_path}...")
stratified_sample_df.toPandas().to_csv(full_path, index=False)
print("File saved successfully!")

## 8. Fetch Labeled Dataset from Google Drive
Fetches the `sampled_dataset_labeled.csv` file from the same `Amazon_2023_EDA` folder in Your Drive.

In [ ]:
import pandas as pd

labeled_file_name = 'sampled_dataset_labeled.csv'
labeled_full_path = os.path.join(drive_folder, labeled_file_name)

if os.path.exists(labeled_full_path):
    print(f"Fetching labeled dataset from: {labeled_full_path}")
    labeled_df = pd.read_csv(labeled_full_path)
    
    print("--- Labeled Dataset Info ---")
    labeled_df.info()
    
    print("\n--- First 5 Rows ---")
    display(labeled_df.head())
else:
    print(f"File not found: {labeled_full_path}. Ensure it exists in the Drive folder.")